# Matching

[All Quick Tutorials](README.md)

Matching uses pretreatment features to pair each observation with one or more similar observations in the other group. HypEx estimates the effect from those pairs. Causal interpretation requires overlap between groups and accounting for factors that influence both assignment and outcome.

## Contents

- [Check baseline imbalance](#check-balance)
- [Estimate the effect](#estimate-effect)
  - [Run matching with one neighbor](#match-one-neighbor)
  - [Read ATE, ATT, ATC, and confidence intervals](#read-effect-report)
  - [Match with three neighbors](#three-neighbors)
- [See what bias adjustment changes](#compare-effects)

<a id="check-balance"></a>

## Check baseline imbalance

Simulate 5,000 observations. Higher values of the historical feature `X_lag_1` increase both the probability of treatment and the outcome. Treatment itself adds exactly 5 units, so $Y = Y(0) + 5T$. Assign `TargetRole` to the historical feature so `HomogeneityTest` compares it across groups.

In [1]:
import numpy as np
import pandas as pd

from hypex import HomogeneityTest, Matching
from hypex.dataset import Dataset, FeatureRole, TargetRole, TreatmentRole

rng = np.random.default_rng(7)
x = rng.uniform(0, 10, size=5_000)
probability = np.clip((x - 2) / 6, 0, 1)
treatment = rng.binomial(1, probability)
true_ate = 5
outcome = 20 + 4 * x + rng.normal(0, 0.4, size=x.size) + true_ate * treatment

frame = pd.DataFrame(
    {"Treatment": treatment, "Target": outcome, "X_lag_1": x}
)
baseline = Dataset(
    roles={"Treatment": TreatmentRole(), "X_lag_1": TargetRole()},
    data=frame[["Treatment", "X_lag_1"]],
)
HomogeneityTest().execute(baseline).summary

,feature,group,control mean,test mean,difference,difference %,ci lower,ci upper,TTest pass,TTest p-value,KSTest pass,KSTest p-value
0,X_lag_1,1,2.803826,7.216167,4.412341,157.368585,4.308997,4.515685,NOT OK,0.0,NOT OK,0.0


Mean `X_lag_1` is about 2.80 in control and 7.22 in treatment. Both the t-test and KS test detect differences. Baseline imbalance therefore contributes to the unadjusted difference in the outcome.

<a id="estimate-effect"></a>

## Estimate the effect

<a id="match-one-neighbor"></a>

### Run matching with one neighbor

Give `X_lag_1` the `FeatureRole` for finding neighbors and assign `TargetRole` to the outcome `Target`. By default, Matching finds one neighbor by Mahalanobis distance and adjusts for remaining feature differences between matched observations.

In [2]:
data = Dataset(
    roles={
        "Treatment": TreatmentRole(),
        "Target": TargetRole(),
        "X_lag_1": FeatureRole(),
    },
    data=frame,
)
matched = Matching().execute(data)
matched.summary

,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,4.7507,0.2384,0.0,4.2834,5.2180
ATC,4.9053,0.2510,0.0,4.4133,5.3973
ATE,4.8294,0.1736,0.0,4.4891,5.1697


<a id="read-effect-report"></a>

### Read ATE, ATT, ATC, and confidence intervals

In the full `summary`, find `Effect Size`, `CI Lower`, and `CI Upper`. `ATE` refers to the full sample, `ATT` to treated units, and `ATC` to control units. The ATE is about 4.83, and its interval includes the true effect of 5.

<a id="three-neighbors"></a>

### Match with three neighbors

Set `n_neighbors=3` to estimate the unobserved outcome by averaging the outcomes of three nearest neighbors. Averaging reduces the influence of any one observation, though the additional neighbors may be less similar. Keep the other settings at their defaults.

In [3]:
matched_three = Matching(n_neighbors=3).execute(data)
matched_three.summary

,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,4.9414,0.1238,0.0,4.6987,5.1841
ATC,4.9698,0.1209,0.0,4.7329,5.2067
ATE,4.9559,0.0870,0.0,4.7854,5.1263


With three neighbors, the ATE is 4.96 and the confidence interval narrows from 4.49–5.17 to 4.79–5.13. In this simulation the estimate moves closer to the true effect of 5.

<a id="compare-effects"></a>

## See what bias adjustment changes

Change only `bias_estimation` to isolate the adjustment's contribution. Display the raw difference in means and the ATE specified by the simulation alongside the unadjusted matching report.

In [4]:
unadjusted = Matching(bias_estimation=False).execute(data)
means = frame.groupby("Treatment")["Target"].mean()
print("Raw difference:", round(means.loc[1] - means.loc[0], 2))
print("Known ATE:", true_ate)
unadjusted.summary

Raw difference: 22.63
Known ATE: 5


,Effect Size,Standard Error,P-value,CI Lower,CI Upper
ATT,6.8592,1.2269,0.0,4.4545,9.2639
ATC,6.6242,1.1780,0.0,4.3154,8.9330
ATE,6.7396,0.8516,0.0,5.0705,8.4088


The raw difference is 22.63. Matching gives an ATE of 6.74 before bias adjustment and 4.83 after it. Outside common support, the adjustment relies on extrapolation. The [in-depth Matching tutorial](Matching.ipynb) covers overlap, matched pairs, and search settings.